# Flower species classification

Classifying a flower photo into one of five species (daisy, dandelion, roses,
sunflowers, tulips) with a fine-tuned MobileNetV2. This notebook goes through the
whole cycle: getting the data, preprocessing, training, evaluation, prediction and
retraining. The same `src/` modules are used by the deployed app, so the notebook
and the app stay in sync.


## Setup

In [ ]:
import sys, random
from pathlib import Path


SRC = Path.cwd().parent / "src" if Path.cwd().name == "notebook" else Path.cwd() / "src"
sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from PIL import Image

import preprocessing, model as model_mod, prediction, data_acquisition
from preprocessing import CLASS_NAMES, get_dataloader, get_transforms

sns.set_theme(style="whitegrid")
DEVICE = model_mod.DEVICE
print("torch", torch.__version__, "| device", DEVICE)
print(CLASS_NAMES)

## Data acquisition

The dataset is the TensorFlow flowers archive. `acquire()` downloads it and splits
each class 70/15/15 into `data/train|val|test`. It skips the download if the splits
already exist.

In [ ]:
data_acquisition.acquire()
data_acquisition.summary()

DATA_DIR = Path(data_acquisition.DATA_ROOT)
TRAIN_DIR, VAL_DIR, TEST_DIR = DATA_DIR/"train", DATA_DIR/"val", DATA_DIR/"test"

## Exploring three features

Looking at three features of the images before modelling: class balance, image
size, and colour per class.

### Class balance

In [ ]:
rows = []
for split, d in [("train", TRAIN_DIR), ("val", VAL_DIR), ("test", TEST_DIR)]:
    for c in CLASS_NAMES:
        rows.append({"split": split, "class": c, "count": len(list((d/c).glob("*.jpg")))})
counts = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=counts, x="class", y="count", hue="split", ax=ax)
ax.set_title("Images per class and split")
plt.tight_layout(); plt.show()
counts.pivot(index="class", columns="split", values="count")

Dandelion and tulip have more photos than daisy, so the set is mildly
imbalanced. That is enough that plain accuracy could be flattered by the bigger
classes, so I report macro-averaged precision, recall and F1 later, which weight
every class the same.

### Image dimensions

In [ ]:
sizes = []
for c in CLASS_NAMES:
    files = list((TRAIN_DIR/c).glob("*.jpg"))
    for f in random.sample(files, min(60, len(files))):
        with Image.open(f) as im:
            sizes.append({"class": c, "width": im.width, "height": im.height})
sizes = pd.DataFrame(sizes)

fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(data=sizes, x="width", y="height", hue="class", alpha=0.6, ax=ax)
ax.set_title("Raw image dimensions (sample)")
plt.tight_layout(); plt.show()
sizes[["width", "height"]].describe()

These are real photos with very different sizes and aspect ratios. The model
needs a fixed input size, so resizing to 224x224 is a required step, not optional.

### Colour per class

In [ ]:
chan = []
for c in CLASS_NAMES:
    files = list((TRAIN_DIR/c).glob("*.jpg"))
    for f in random.sample(files, min(60, len(files))):
        with Image.open(f) as im:
            arr = np.asarray(im.convert("RGB")) / 255.0
        chan.append({"class": c, "R": arr[...,0].mean(), "G": arr[...,1].mean(), "B": arr[...,2].mean()})
chan = pd.DataFrame(chan)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, ch in zip(axes, ["R", "G", "B"]):
    sns.boxplot(data=chan, x="class", y=ch, ax=ax)
    ax.set_title(f"mean {ch}"); ax.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()
chan.groupby("class")[["R","G","B"]].mean()

Yellow flowers (sunflowers, dandelions) score high on red and green, roses
carry more red than green. But the boxes overlap a lot, so colour alone can't tell
a yellow tulip from a dandelion. That is why I use a CNN that also picks up petal
shape and texture rather than a simple colour rule.

### Sample images

In [ ]:
fig, axes = plt.subplots(len(CLASS_NAMES), 4, figsize=(11, 12))
for i, c in enumerate(CLASS_NAMES):
    for j, f in enumerate(list((TRAIN_DIR/c).glob("*.jpg"))[:4]):
        axes[i, j].imshow(Image.open(f)); axes[i, j].axis("off")
        if j == 0: axes[i, j].set_title(c, loc="left")
plt.tight_layout(); plt.show()

## Preprocessing

Defined once in `src/preprocessing.py`. Resize to 224x224, and on the training set
add random flips, rotation and colour jitter as augmentation to reduce overfitting.
Everything is normalized with ImageNet stats because the backbone is pretrained on
ImageNet. Validation, test and inference skip the augmentation.

In [ ]:
BATCH = 32
train_loader = get_dataloader(TRAIN_DIR, batch_size=BATCH, train=True)
val_loader   = get_dataloader(VAL_DIR,   batch_size=BATCH, train=False)
test_loader  = get_dataloader(TEST_DIR,  batch_size=BATCH, train=False)

print(get_transforms(train=True))
xb, yb = next(iter(train_loader))
print(xb.shape, yb[:8].tolist())

## Model

MobileNetV2 pretrained on ImageNet, with its classifier head replaced by a dropout
layer and a linear layer for the 5 classes. Dropout for regularization, the
pretrained backbone so I'm not training from scratch.

In [ ]:
net = model_mod.build_model(pretrained=True)
total = sum(p.numel() for p in net.parameters())
print(f"params: {total:,}")
print(net.classifier)

## Training

Adam optimizer, L2 weight decay, and early stopping on the validation loss (keeps
the best weights).

In [ ]:
history = model_mod.train_model(net, train_loader, val_loader,
                                epochs=15, lr=1e-3, weight_decay=1e-4, patience=3)

In [ ]:
hist = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist["train_loss"], label="train"); axes[0].plot(hist["val_loss"], label="val")
axes[0].set_title("loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(hist["train_acc"], label="train"); axes[1].plot(hist["val_acc"], label="val")
axes[1].set_title("accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

## Evaluation

Accuracy, loss, precision, recall and F1 on the test set, plus a per-class report
and a confusion matrix.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)

criterion = torch.nn.CrossEntropyLoss()
test_loss, _ = model_mod.evaluate_loss(net, test_loader, criterion)
y_true, y_pred = model_mod.predict_loader(net, test_loader)

metrics = {
    "accuracy":  accuracy_score(y_true, y_pred),
    "loss":      test_loss,
    "precision": precision_score(y_true, y_pred, average="macro"),
    "recall":    recall_score(y_true, y_pred, average="macro"),
    "f1":        f1_score(y_true, y_pred, average="macro"),
}
for k, v in metrics.items():
    print(f"{k:10s} {v:.4f}")

In [ ]:
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.tight_layout(); plt.show()

The per-class precision and recall stay close to each other, so the model
isn't trading one class off against another. Most of the remaining confusion is
between the two yellow classes (sunflowers and dandelion), which lines up with the
colour overlap seen earlier.

## Saving the model

The checkpoint stores the weights and the class names, so the app can load it
without needing this notebook.

In [ ]:
MODEL_PATH = DATA_DIR.parent / "models" / "flowers_model.pth"
model_mod.save_model(net, MODEL_PATH)
print("saved", MODEL_PATH, f"{MODEL_PATH.stat().st_size/1e6:.1f} MB")

## Prediction

`prediction.predict` is the same function the app calls. Running it on a test image
and checking the label matches the folder it came from.

In [ ]:
sample_class = "sunflowers"
sample_img = next((TEST_DIR/sample_class).glob("*.jpg"))
prediction.clear_model_cache()
result = prediction.predict(sample_img, model_path=MODEL_PATH)

plt.imshow(Image.open(sample_img).convert("RGB")); plt.axis("off")
plt.title(f"true {sample_class} | pred {result['label']} ({result['confidence']:.1%})")
plt.show()
for label, p in result["probabilities"].items():
    print(f"{label:12s} {p:.1%}")

## Retraining

Retraining loads the saved model and continues training on it, rather than starting
over. In the app this runs when a user uploads new images. Here I fine-tune a few
epochs on the training data and save over the checkpoint, which is what the app's
retrain button does.

In [ ]:
retrained, _ = model_mod.retrain_model(MODEL_PATH, train_loader, val_loader,
                                       epochs=4, lr=5e-4, patience=2)
_, rt_acc = model_mod.evaluate_loss(retrained, test_loader, criterion)
print(f"test accuracy after retraining: {rt_acc:.3f}")

model_mod.save_model(retrained, MODEL_PATH)
print("saved retrained model")